# HAKE-MER — M2 control: sans encodeur inter-émotions

**Backbone:** DistilBERT-base · **Modules:** M1 + M2 with phrase cross-attention, **no** Transformer over the 28 emotion vectors.

**Protocol:** batch 16, LR 5e-5, 4 epochs, seeds 42 / 123 / 456 (same as main ladder).

Compare test F1-macro to **M1+M2 reference: 0.505 ± 0.004**.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
628af5c


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.3 MB/s eta 0:00:00


In [4]:
!./run_m1_m2_controls_campaign.sh --variant no_enc --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 4.97MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 616kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 1.14MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.23MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 55.4MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:  17% 469k/2.77M [00:00<00:01, 1.51MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes:  94% 2.60M/2.77M [00:00<00:00, 7.76MB/s,   ???B/s  ]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 5.27MB/s,  267kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 5.34MB/s,  273kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:  58% 203k/350k [00:00<00:00, 699kB/s,   ???B/s  ]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 848kB/s, 34.0kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing f

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_no_enc_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_m2_no_enc_distilbert_base_uncased_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  f1_micro: 0.5735 ± 0.0024
  f1_macro: 0.4761 ± 0.0132
  exact_match: 0.4489 ± 0.0045
  map: 0.5028 ± 0.0016


In [6]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_m2_no_enc_{slug}_campaign.json")
out_name = "m1_m2_no_enc_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_no_enc/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download m1_m2_no_enc_distilbert_step0.zip (7.3 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Archive

1. **File → Download → .ipynb** (executed) → send with the zip.
2. Unzip into `reference/artifacts/` on your machine (or send both to your agent).